
# 練習問題: 行列積をマルチコア + SIMD で高速化する (総仕上げ)

## 目標

行列積 `C = A * B` という現実的なカーネルを, **マルチコア並列 (parallel for) と SIMD (omp simd) の両方**で高速化する. これまで別々に学んだ2つの並列化を1つのカーネルで組み合わせる, この授業の総仕上げである.

ループは i-k-j 順とし, 外側 `i` ループは行ごとに独立なのでスレッド並列に, 最内 `j` ループは連続したメモリへの積和なのでSIMD化に向いている.

## 課題

`matmul.cpp` (または `matmul.f90`) では, 外側 `i` ループの `#pragma omp parallel for` (Fortran は `!$omp parallel do`) は **既に与えられている**.
コメント `TODO` の指示に従い, **最内 `j` ループをSIMD化する指示行を1つ追加**せよ.

- C++: 最内 `for (j ...)` の直前に `#pragma omp simd` を1行加える.
- Fortran: 最内 `do j` の直前に `!$omp simd` を1行加える.

これで「外: マルチコア並列」「内: SIMD」という二重の並列化になる.

## コンパイルと実行

`parallel for` と `omp simd` の両方を使うので `-mp=multicore` が必要.

```
# C++
nvc++ -fast -mp=multicore matmul.cpp -o matmul_cpp.exe
# Fortran
nvfortran -fast -mp=multicore matmul.f90 -o matmul_f90.exe

# スレッド数を変えて GFLOPS を測る
OMP_PROC_BIND=true OMP_NUM_THREADS=1  ./matmul_cpp.exe
OMP_PROC_BIND=true OMP_NUM_THREADS=8  ./matmul_cpp.exe
```

`n` はコマンドライン引数で指定できる (既定 1024). `A[i]=1`, `B[i]=2` としているので各要素は `2*n` になり, `check: OK` で正しさを確認できる.

## 期待される結果

- `n=1024 : XX.XXX GFLOPS  (check: OK)` のように表示される.
- `OMP_NUM_THREADS` を `1, 2, 4, 8, ...` と増やすと GFLOPS が伸びる (台数効果). `OMP_PROC_BIND=true` でスレッドをコアに固定すると安定する.
- 05_speedup の matmul はマルチコア化のみ, 本問はそこに SIMD を重ねたものである. GPU 版の行列積 (10 番台) とも比較し, CPU のピーク性能 (コア数 × SIMD幅 × FMA × クロック) にどこまで近づけるか考えてみよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ matmul.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* C = A * B (いずれも n x n 行列, 行優先). i-k-j 順のループ. */
void matmul(long n, double * A, double * B, double * C) {
  for (long i = 0; i < n; i++)
    for (long j = 0; j < n; j++) C[i * n + j] = 0.0;

  #pragma omp parallel for
  for (long i = 0; i < n; i++) {
    for (long k = 0; k < n; k++) {
      double a = A[i * n + k];
      // TODO: 最内 j ループを omp simd でSIMD化せよ (下の for の直前に1行追加).
      for (long j = 0; j < n; j++) {
        C[i * n + j] += a * B[k * n + j];
      }
    }
  }
}

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 1024);
  double * A = (double *)malloc(sizeof(double) * n * n);
  double * B = (double *)malloc(sizeof(double) * n * n);
  double * C = (double *)malloc(sizeof(double) * n * n);
  for (long i = 0; i < n * n; i++) { A[i] = 1.0; B[i] = 2.0; }

  double t0 = omp_get_wtime();
  matmul(n, A, B, C);
  double dt = omp_get_wtime() - t0;

  double gflops = 2.0 * (double)n * n * n / dt * 1e-9;
  /* A[i]=1, B[i]=2 なので C の各要素は 2*n になるはず. checksum で確認. */
  double expected = 2.0 * (double)n;
  long err = 0;
  for (long i = 0; i < n * n; i++) if (C[i] != expected) err++;
  printf("n=%ld : %.3f GFLOPS  (check: %s)\n", n, gflops, err == 0 ? "OK" : "NG");
  free(A); free(B); free(C);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore matmul.cpp -o matmul_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matmul_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ matmul.f90
! C = A * B (いずれも n x n 行列). i-k-j 順のループ.
program matmul
  use omp_lib
  implicit none
  character(len=32) :: arg
  integer(8) :: n, i, j, k, err
  real(8), allocatable :: A(:,:), B(:,:), C(:,:)
  real(8) :: a_ik, t0, dt, gflops, expected
  n = 1024
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(A(n,n), B(n,n), C(n,n))
  A = 1.0d0; B = 2.0d0; C = 0.0d0

  t0 = omp_get_wtime()
  !$omp parallel do private(j,k,a_ik)
  do i = 1, n
     do k = 1, n
        a_ik = A(i,k)
        ! TODO: 最内 j ループを omp simd でSIMD化せよ (下の do の直前に1行追加).
        do j = 1, n
           C(i,j) = C(i,j) + a_ik * B(k,j)
        end do
     end do
  end do
  dt = omp_get_wtime() - t0

  gflops = 2.0d0 * real(n, 8) * real(n, 8) * real(n, 8) / dt * 1e-9
  ! A=1, B=2 なので C の各要素は 2*n になるはず.
  expected = 2.0d0 * real(n, 8)
  err = 0
  do j = 1, n
     do i = 1, n
        if (C(i,j) /= expected) err = err + 1
     end do
  end do
  if (err == 0) then
     print "(a,i0,a,f0.3,a)", "n=", n, " : ", gflops, " GFLOPS  (check: OK)"
  else
     print "(a,i0,a,f0.3,a)", "n=", n, " : ", gflops, " GFLOPS  (check: NG)"
  end if
end program matmul

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore matmul.f90 -o matmul_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matmul_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:matmul.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:matmul.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:matmul.cpp}